# Full platform demo

End-to-end run of the urban data platform:

1. Download raw data and bronze-ingest
2. Silver promote + gold integrate (validate along the way)
3. Run analytical queries Q1–Q6
4. Spark query optimizations (cache, partition pruning, broadcast, AQE)
5. Build gold data products
6. Benchmark products vs on-demand / AQE
7. Generate incremental updates (schema evolution) and re-ingest
8. Show monitoring (`pipeline_runs`)
9. Answer ops Spark SQL questions (failures, duration, rejects, trends)
10. Production readiness metrics (incremental / analytical / storage / validation / monitoring overhead)

Prefer the project `.venv` Jupyter kernel. Set flags in the setup cell as needed.


## 0. Setup

In [1]:
import sys
import time
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.spark import create_spark, ensure_runtime

ensure_runtime()

PosixPath('/Users/juozas/Documents/Projects/kth/id2221_labs/id2221-labs')

## 1. Download raw data and bronze ingest

In [2]:
from src.bronze.download import download_raw
from src.bronze.ingest import ingest_all
from src.lake import RAW

download_raw()

spark = create_spark("run-pipeline")
print("\n=== BRONZE ingest ===")
ingest_all(spark)

/Users/juozas/Documents/Projects/kth/id2221_labs/id2221-labs/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
Retrieving folder contents


Processing file 1NGT8RR-NtBf-4xxII4pfwBanE_BgBvoW air_quality.zip
Processing file 1-gO-MhXTIPbHRkyJQo8nTxly72gT3r9t taxi_zone_lookup.csv
Processing file 1q-Lw24XFqJ42XSJRuUV_ced3aJ6kKQ2M weather.csv
Processing file 17v0eFEontYEKtGoqyB0v9rj_BEDc7snA yellow_tripdata_2024-01.parquet
Processing file 1N-dRuGdd_lOYGAbdbgJJWMyIsV_lz057 yellow_tripdata_2024-02.parquet
Processing file 1oUxC0cLWqOatddyT8aB06VvFFZY0-lu2 yellow_tripdata_2024-03.parquet


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1NGT8RR-NtBf-4xxII4pfwBanE_BgBvoW
From (redirected): https://drive.google.com/uc?id=1NGT8RR-NtBf-4xxII4pfwBanE_BgBvoW&confirm=t&uuid=df13a01a-a407-4d43-9f2f-7e7e7d7f2a63
To: /Users/juozas/Documents/Projects/kth/id2221_labs/id2221-labs/data/drive_download/air_quality.zip
100%|██████████| 66.3M/66.3M [00:01<00:00, 36.4MB/s]
Downloading...
From: https://drive.google.com/uc?id=1-gO-MhXTIPbHRkyJQo8nTxly72gT3r9t
To: /Users/juozas/Documents/Projects/kth/id2221_labs/id2221-labs/data/drive_download/taxi_zone_lookup.csv
100%|██████████| 12.3k/12.3k [00:00<00:00, 24.1MB/s]
Downloading...
From: https://drive.google.com/uc?id=1q-Lw24XFqJ42XSJRuUV_ced3aJ6kKQ2M
To: /Users/juozas/Documents/Projects/kth/id2221_labs/id2221-labs/data/drive_download/weather.csv
100%|██████████| 1.07M/1.07M [00:00<00:00, 17.6MB/s]
Downloading...
From: https:

unzip data/drive_download/air_quality.zip -> data/raw/air_quality
copy  yellow_tripdata_2024-03.parquet -> data/raw/taxi_trips/yellow
copy  taxi_zone_lookup.csv -> data/raw/taxi_zones
copy  weather.csv -> data/raw/weather
copy  yellow_tripdata_2024-02.parquet -> data/raw/taxi_trips/yellow
copy  yellow_tripdata_2024-01.parquet -> data/raw/taxi_trips/yellow
:: loading settings :: url = jar:file:/Users/juozas/Documents/Projects/kth/id2221_labs/id2221-labs/.venv/lib/python3.9/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/juozas/.ivy2/cache
The jars for the packages stored in: /Users/juozas/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-10cd0941-1b6e-42ac-9c54-9d629b38767f;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.1 in central
	found io.delta#delta-storage;3.2.1 in central
	found org.antlr#antlr4-runtime;4.9.3 in local-m2-cache
:: resolution report :: resolve 180ms :: artifacts dl 4ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.1 from central in [default]
	io.delta#delta-storage;3.2.1 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from local-m2-cache in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default   

...

26/09/26 14:40:48 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


[bronze/air_quality] written (117,438 rows)


[taxi_trips/yellow] schema ok=True


[bronze/taxi_trips] written (9,554,778 rows)
[taxi_zones] schema ok=True
[bronze/taxi_zones] written (265 rows)
[weather] schema ok=True


[bronze/weather] written (8,784 rows)


## 2. Silver promote + gold integrate

In [3]:
from src.silver.promote import promote_all
from src.gold.integrate import integrate
from src.benchmark.storage import compare_integrated_layouts
from src.lake import GOLD, SILVER, read_delta, show_delta

print("=== SILVER ===")
promote_all(spark)

print("\nSilver tables")
for name in ["taxi_zones", "weather", "air_quality", "taxi_trips"]:
    show_delta(spark, SILVER / name, n=1)

print("\n=== GOLD integrate ===")
integrate(spark)
compare_integrated_layouts()

=== SILVER ===


[silver/air_quality] kept=112,838  rejected=4,600


[silver/taxi_trips] kept=9,417,383  rejected=137,395
[silver/taxi_zones] kept=265  rejected=0


[silver/weather] kept=8,784  rejected=0

Silver tables
taxi_zones: 265 rows @ data/lake/silver/taxi_zones
+-----------+-------+--------------+------------+
|location_id|borough|zone          |service_zone|
+-----------+-------+--------------+------------+
|1          |EWR    |Newark Airport|EWR         |
+-----------+-------+--------------+------------+
only showing top 1 row

weather: 8784 rows @ data/lake/silver/weather
+-----------+-------------------+-------------+------------------+----------------+----------------+
|station_id |obs_timestamp      |temperature_c|wind_speed_ms     |observation_date|observation_hour|
+-----------+-------------------+-------------+------------------+----------------+----------------+
|72505394728|2024-05-01 00:00:00|11.1         |11.305555555555555|2024-05-01      |0               |
+-----------+-------------------+-------------+------------------+----------------+----------------+
only showing top 1 row

air_quality: 112838 rows @ data/lake/silver/a

write by_date: 8.5s
integrated_taxi_trips: 9417383 rows @ data/lake/gold/integrated_taxi_trips
+---------+---------+-------------------+-------------------+---------------+-------------+------------------+----------------------------+--------------+-------------------+-------------------+---------------+-----------+----------+------------+------------+-------------+-----------------+-----------------+---------------------------+-----------+-----------+
|taxi_type|vendor_id|pickup_datetime    |dropoff_datetime   |passenger_count|trip_distance|pickup_location_id|pickup_zone                 |pickup_borough|dropoff_location_id|dropoff_zone       |dropoff_borough|fare_amount|tip_amount|tolls_amount|total_amount|temperature_c|wind_speed_ms    |pm25             |pm25_unit                  |pickup_date|pickup_hour|
+---------+---------+-------------------+-------------------+---------------+-------------+------------------+----------------------------+--------------+-------------------+-------

write by_borough: 8.0s
Storage
integrated_taxi_trips                    files=  222  partitions=  92  size=   227.7 MB
integrated_taxi_trips_by_borough         files=  200  partitions=   8  size=   226.9 MB


## 3. Analytical queries Q1–Q6

In [4]:
from src.queries.analytical import (
    QUERY_1_MONTHLY_ZONE_DEMAND,
    QUERY_2_WEATHER_DISTANCE,
    QUERY_3_PM25_DEMAND,
    QUERY_4_ZONE_WEATHER_SENSITIVITY,
    QUERY_5_PEAK_HOURS_BY_DOW,
    QUERY_6_MONTHLY_TRENDS,
)

read_delta(spark, GOLD / "integrated_taxi_trips").createOrReplaceTempView(
    "integrated_taxi_trips"
)

for name, sql in [
    ("Q1 monthly zone demand", QUERY_1_MONTHLY_ZONE_DEMAND),
    ("Q2 weather distance", QUERY_2_WEATHER_DISTANCE),
    ("Q3 pm25 demand", QUERY_3_PM25_DEMAND),
    ("Q4 zone weather sensitivity", QUERY_4_ZONE_WEATHER_SENSITIVITY),
    ("Q5 peak hours by DOW", QUERY_5_PEAK_HOURS_BY_DOW),
    ("Q6 monthly trends", QUERY_6_MONTHLY_TRENDS),
]:
    print(f"\n=== {name} ===")
    spark.sql(sql).show(15, truncate=False)


=== Q1 monthly zone demand ===


+----------+------------------+--------------+----------------------------+-----------+-----------+---------------+
|trip_month|pickup_location_id|pickup_borough|pickup_zone                 |total_trips|active_days|avg_daily_trips|
+----------+------------------+--------------+----------------------------+-----------+-----------+---------------+
|2024-01-01|161               |Manhattan     |Midtown Center              |141738     |31         |4572.19        |
|2024-01-01|237               |Manhattan     |Upper East Side South       |141263     |31         |4556.87        |
|2024-01-01|132               |Queens        |JFK Airport                 |141159     |31         |4553.52        |
|2024-01-01|236               |Manhattan     |Upper East Side North       |135334     |31         |4365.61        |
|2024-01-01|162               |Manhattan     |Midtown East                |105466     |31         |3402.13        |
|2024-01-01|230               |Manhattan     |Times Sq/Theatre District 

+-----------------------+-----------------------+----------+------------------+
|temp_category          |wind_category          |trip_count|avg_distance_miles|
+-----------------------+-----------------------+----------+------------------+
|Cold (0°C to 10°C)     |Calm (<2 m/s)          |395811    |3.29              |
|Cold (0°C to 10°C)     |High Wind (>6 m/s)     |2272210   |3.31              |
|Cold (0°C to 10°C)     |Moderate Wind (2-6 m/s)|4128105   |3.29              |
|Freezing (<0°C)        |Calm (<2 m/s)          |13866     |4.2               |
|Freezing (<0°C)        |High Wind (>6 m/s)     |478247    |3.23              |
|Freezing (<0°C)        |Moderate Wind (2-6 m/s)|622618    |3.2               |
|Moderate (10°C to 20°C)|Calm (<2 m/s)          |117228    |3.37              |
|Moderate (10°C to 20°C)|High Wind (>6 m/s)     |460189    |3.28              |
|Moderate (10°C to 20°C)|Moderate Wind (2-6 m/s)|669868    |3.41              |
|Warm (>20°C)           |Calm (<2 m/s)  

+----------+-----+--------------+--------------+
|pm25_level|trips|observed_hours|trips_per_hour|
+----------+-----+--------------+--------------+
|34.0      |4188 |1             |4188.0        |
|33.0      |9071 |2             |4535.5        |
|31.0      |5019 |1             |5019.0        |
|30.0      |18438|3             |6146.0        |
|29.0      |18240|2             |9120.0        |
|28.0      |17702|3             |5900.67       |
|27.0      |14835|2             |7417.5        |
|26.0      |66047|10            |6604.7        |
|25.0      |18435|3             |6145.0        |
|24.0      |54765|10            |5476.5        |
|23.0      |20069|5             |4013.8        |
|22.0      |54775|12            |4564.58       |
|21.0      |80355|17            |4726.76       |
|20.0      |83362|16            |5210.13       |
|19.0      |79234|17            |4660.82       |
+----------+-----+--------------+--------------+
only showing top 15 rows


=== Q4 zone weather sensitivity ===


+-----------------------------+-------+------+------+-------+-------------+
|pickup_zone                  |coldest|cool  |warm  |warmest|pct_variation|
+-----------------------------+-------+------+------+-------+-------------+
|Financial District South     |16.25  |13.0  |12.66 |10.69  |42.28        |
|Financial District North     |26.62  |21.65 |21.07 |18.24  |38.27        |
|Meatpacking/West Village West|49.81  |38.38 |40.34 |34.71  |37.0         |
|Lower East Side              |58.06  |45.25 |48.38 |41.47  |34.35        |
|Little Italy/NoLiTa          |52.69  |41.92 |43.93 |38.29  |32.57        |
|Greenwich Village South      |75.86  |62.62 |63.19 |57.27  |28.72        |
|West Village                 |121.51 |101.08|101.63|92.03  |28.33        |
|Battery Park City            |31.51  |28.26 |26.96 |23.97  |27.24        |
|TriBeCa/Civic Center         |66.51  |58.87 |57.36 |50.75  |27.0         |
|World Trade Center           |25.62  |21.19 |21.69 |19.84  |26.17        |
|East Villag

+-----------+---------+----------+-----------+---------+
|day_of_week|peak_hour|trip_count|active_days|avg_trips|
+-----------+---------+----------+-----------+---------+
|Monday     |18       |80482     |13         |6190.92  |
|Tuesday    |18       |99614     |13         |7662.62  |
|Wednesday  |18       |110914    |13         |8531.85  |
|Thursday   |18       |119124    |13         |9163.38  |
|Friday     |18       |102921    |13         |7917.0   |
|Saturday   |19       |96485     |13         |7421.92  |
|Sunday     |0        |79601     |13         |6123.15  |
+-----------+---------+----------+-----------+---------+


=== Q6 monthly trends ===


26/09/26 14:43:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/26 14:43:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/26 14:43:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+----------+-----------+-----------+---------------+--------------------+------------------------+
|trip_month|total_trips|active_days|avg_daily_trips|prev_month_avg_daily|pct_change_vs_prev_month|
+----------+-----------+-----------+---------------+--------------------+------------------------+
|2024-01-01|2926910    |31         |94416.45       |NULL                |NULL                    |
|2024-02-01|2966705    |29         |102300.17      |94416.45            |8.35                    |
|2024-03-01|3523766    |31         |113669.87      |102300.17           |11.11                   |
|2024-04-01|2          |1          |2.0            |113669.87           |-100.0                  |
+----------+-----------+-----------+---------------+--------------------+------------------------+



26/09/26 14:43:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/26 14:43:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/26 14:43:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/26 14:43:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/26 14:43:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/26 14:43:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


## 4. Query optimizations (on silver)

Cache reuse, partition pruning, broadcast joins, and AQE — same experiments as `query_optimization.ipynb`.

In [5]:
from pyspark.sql.functions import avg, broadcast


def benchmark_and_verify(df_baseline, df_optimized, name="Optimization Test"):
    print(f"=== {name} ===")
    print("\n--- Baseline Plan ---")
    df_baseline.explain("formatted")
    print("\n--- Optimized Plan ---")
    df_optimized.explain("formatted")

    t0 = time.time()
    base_count = df_baseline.count()
    base_time = time.time() - t0

    t0 = time.time()
    opt_count = df_optimized.count()
    opt_time = time.time() - t0

    assert base_count == opt_count, f"Mismatch! {base_count} vs {opt_count}"
    if base_count < 10000:
        assert (
            df_baseline.orderBy(*df_baseline.columns).collect()
            == df_optimized.orderBy(*df_optimized.columns).collect()
        )

    speedup = ((base_time - opt_time) / base_time) * 100 if base_time > 0 else 0
    print(f"Row Count:      {base_count:,}")
    print(f"Baseline Time:  {base_time:.2f}s")
    print(f"Optimized Time: {opt_time:.2f}s")
    print(f"Speedup:        {speedup:.1f}%\n")


trips = read_delta(spark, SILVER / "taxi_trips")
weather = read_delta(spark, SILVER / "weather")
zones = read_delta(spark, SILVER / "taxi_zones")

# --- Cache ---
weather_hourly = weather.groupBy("observation_date", "observation_hour").agg(
    avg("temperature_c").alias("temperature_c")
)
intermediate_df = (
    trips.filter("fare_amount > 0 AND trip_distance > 0")
    .select("pickup_date", "pickup_hour", "fare_amount", "trip_distance")
    .join(
        broadcast(weather_hourly),
        (trips.pickup_date == weather_hourly.observation_date)
        & (trips.pickup_hour == weather_hourly.observation_hour),
        "inner",
    )
)

t0 = time.time()
c1 = intermediate_df.groupBy("pickup_hour").avg("fare_amount").count()
c2 = intermediate_df.groupBy("temperature_c").avg("trip_distance").count()
base_time = time.time() - t0

t0 = time.time()
cached_df = intermediate_df.cache()
cached_df.count()
o1 = cached_df.groupBy("pickup_hour").avg("fare_amount").count()
o2 = cached_df.groupBy("temperature_c").avg("trip_distance").count()
opt_time = time.time() - t0
cached_df.unpersist()
assert c1 == o1 and c2 == o2
print("=== Cache Performance ===")
print(f"Baseline (uncached, 2 queries): {base_time:.2f}s")
print(f"Optimized (cached, 2 queries):  {opt_time:.2f}s")
print(f"Speedup: {((base_time - opt_time) / base_time) * 100:.1f}%\n")

# --- Partition pruning ---
df_base = trips.filter("date(pickup_datetime) = '2024-01-01'")
df_opt = trips.filter(
    "pickup_date = '2024-01-01' AND date(pickup_datetime) = '2024-01-01'"
)
benchmark_and_verify(df_base, df_opt, "Partition Pruning")

# --- Broadcast join ---
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
df_base = trips.join(zones, trips.pickup_location_id == zones.location_id)
df_opt = trips.join(broadcast(zones), trips.pickup_location_id == zones.location_id)
benchmark_and_verify(df_base, df_opt, "Broadcast Join")

# --- AQE ---
query_df = trips.groupBy("pickup_location_id", "pickup_hour").agg(
    {"fare_amount": "avg", "trip_distance": "sum"}
)
spark.conf.set("spark.sql.adaptive.enabled", "false")
t0 = time.time()
n_off = query_df.count()
t_off = time.time() - t0

spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
t0 = time.time()
n_on = query_df.count()
t_on = time.time() - t0
assert n_off == n_on
print("=== AQE Performance ===")
print(f"AQE Disabled: {t_off:.2f}s")
print(f"AQE Enabled:  {t_on:.2f}s")
print(f"Speedup:      {((t_off - t_on) / t_off) * 100:.1f}%\n")

=== Cache Performance ===
Baseline (uncached, 2 queries): 4.68s
Optimized (cached, 2 queries):  3.35s
Speedup: 28.4%

=== Partition Pruning ===

--- Baseline Plan ---
== Physical Plan ==
* Project (4)
+- * Filter (3)
   +- * ColumnarToRow (2)
      +- Scan parquet  (1)


(1) Scan parquet 
Output [14]: [taxi_type#31769, vendor_id#31770, pickup_datetime#31771, dropoff_datetime#31772, passenger_count#31773, trip_distance#31774, pickup_location_id#31775, dropoff_location_id#31776, fare_amount#31777, tip_amount#31778, tolls_amount#31779, total_amount#31780, pickup_hour#31782, pickup_date#31781]
Batched: true
Location: PreparedDeltaFileIndex [file:/Users/juozas/Documents/Projects/kth/id2221_labs/id2221-labs/data/lake/silver/taxi_trips]
PushedFilters: [IsNotNull(pickup_datetime), GreaterThanOrEqual(pickup_datetime,2024-01-01 01:00:00.0), LessThan(pickup_datetime,2024-01-02 01:00:00.0)]
ReadSchema: struct<taxi_type:string,vendor_id:int,pickup_datetime:timestamp,dropoff_datetime:timestamp,passe

Row Count:      9,417,383
Baseline Time:  1.77s
Optimized Time: 0.82s
Speedup:        53.6%



=== AQE Performance ===
AQE Disabled: 0.94s
AQE Enabled:  0.45s
Speedup:      52.2%



## 5. Gold data products

Same builders as `data_products.ipynb`.

In [6]:
from src.gold.products import build_products

print("=== GOLD data products ===")
build_products(spark, force=False)

=== GOLD data products ===
--- Pipeline Execution Plan (Schema Version: 1.0) ---
[REFRESHING] Product: daily_borough_mobility...


Product daily_borough_mobility refreshed successfully.
[REFRESHING] Product: taxi_zone_monthly_demand...


Product taxi_zone_monthly_demand refreshed successfully.
[REFRESHING] Product: weather_impact_summary...


Product weather_impact_summary refreshed successfully.
[REFRESHING] Product: air_quality_demand_summary...


Product air_quality_demand_summary refreshed successfully.
[REFRESHING] Product: zone_weather_sensitivity...


Product zone_weather_sensitivity refreshed successfully.


## 6. Benchmarks (products vs on-demand + AQE)

Same evaluations as `benchmark.ipynb`.

In [7]:
from src.benchmark.evaluate import evaluate_query
from src.queries.analytical import (
    QUERY_1_MONTHLY_ZONE_DEMAND as query_1,
    QUERY_2_WEATHER_DISTANCE as query_2,
    QUERY_3_PM25_DEMAND as query_3,
    QUERY_4_ZONE_WEATHER_SENSITIVITY as query_4,
    QUERY_5_PEAK_HOURS_BY_DOW as query_5,
    QUERY_6_MONTHLY_TRENDS as query_6,
)

read_delta(spark, GOLD / "integrated_taxi_trips").createOrReplaceTempView(
    "integrated_taxi_trips"
)
prod = GOLD / "data_products"

evaluate_query(
    spark,
    "Query 1: Monthly Taxi Demand per Zone",
    spark.sql(query_1),
    read_delta(spark, prod / "taxi_zone_monthly_demand").select(
        "trip_month", "pickup_location_id", "pickup_borough", "pickup_zone",
        "total_trips", "active_days", "avg_daily_trips",
    ),
    prod / "taxi_zone_monthly_demand",
)
evaluate_query(
    spark,
    "Query 2: Average Distance by Weather",
    spark.sql(query_2),
    read_delta(spark, prod / "weather_impact_summary").select(
        "temp_category", "wind_category", "trip_count", "avg_distance_miles"
    ),
    prod / "weather_impact_summary",
)
evaluate_query(
    spark,
    "Query 3: Air Quality vs Demand",
    spark.sql(query_3),
    read_delta(spark, prod / "air_quality_demand_summary").select(
        "pm25_level", "trips", "observed_hours", "trips_per_hour"
    ),
    prod / "air_quality_demand_summary",
)
evaluate_query(
    spark,
    "Query 4: Zone Weather Demand Variation",
    spark.sql(query_4),
    read_delta(spark, prod / "zone_weather_sensitivity").select(
        "pickup_zone", "coldest", "cool", "warm", "warmest", "pct_variation"
    ),
    prod / "zone_weather_sensitivity",
)
evaluate_query(spark, "Query 5: Peak Travel Hours by Day of Week", aqe_query=query_5)
evaluate_query(spark, "Query 6: Monthly Trends in Taxi Demand", aqe_query=query_6)


EVALUATING: Query 1: Monthly Taxi Demand per Zone

--- Baseline Physical Plan (On-Demand) ---
== Physical Plan ==
AdaptiveSparkPlan (12)
+- Sort (11)
   +- Exchange (10)
      +- HashAggregate (9)
         +- Exchange (8)
            +- HashAggregate (7)
               +- HashAggregate (6)
                  +- Exchange (5)
                     +- HashAggregate (4)
                        +- Project (3)
                           +- Filter (2)
                              +- Scan parquet  (1)


(1) Scan parquet 
Output [4]: [pickup_location_id#42434, pickup_zone#42435, pickup_borough#42436, pickup_date#42448]
Batched: true
Location: PreparedDeltaFileIndex [file:/Users/juozas/Documents/Projects/kth/id2221_labs/id2221-labs/data/lake/gold/integrated_taxi_trips]
PartitionFilters: [isnotnull(pickup_date#42448)]
PushedFilters: [IsNotNull(pickup_zone), Not(EqualTo(pickup_zone,UNKNOWN))]
ReadSchema: struct<pickup_location_id:int,pickup_zone:string,pickup_borough:string>

(2) Filter
Input [4]:


[VERIFIED] Optimized query produces identical results to baseline.

--- Performance Summary ---
Result Rows:       775
Baseline Time:     1.06s
Optimized Time:    0.29s
Speedup:           73.0%
Storage Overhead:  33.9 KB (4 files)


EVALUATING: Query 2: Average Distance by Weather

--- Baseline Physical Plan (On-Demand) ---
== Physical Plan ==
AdaptiveSparkPlan (9)
+- Sort (8)
   +- Exchange (7)
      +- HashAggregate (6)
         +- Exchange (5)
            +- HashAggregate (4)
               +- Project (3)
                  +- Filter (2)
                     +- Scan parquet  (1)


(1) Scan parquet 
Output [4]: [trip_distance#42433, temperature_c#42444, wind_speed_ms#42445, pickup_date#42448]
Batched: true
Location: PreparedDeltaFileIndex [file:/Users/juozas/Documents/Projects/kth/id2221_labs/id2221-labs/data/lake/gold/integrated_taxi_trips]
PushedFilters: [IsNotNull(trip_distance), GreaterThan(trip_distance,0.0), LessThan(trip_distance,100.0)]
ReadSchema: struct<trip_distance:double


[VERIFIED] Optimized query produces identical results to baseline.

--- Performance Summary ---
Result Rows:       12
Baseline Time:     0.74s
Optimized Time:    0.21s
Speedup:           71.9%
Storage Overhead:  3.3 KB (1 files)


EVALUATING: Query 3: Air Quality vs Demand

--- Baseline Physical Plan (On-Demand) ---
== Physical Plan ==
AdaptiveSparkPlan (12)
+- Sort (11)
   +- Exchange (10)
      +- HashAggregate (9)
         +- Exchange (8)
            +- HashAggregate (7)
               +- HashAggregate (6)
                  +- Exchange (5)
                     +- HashAggregate (4)
                        +- Project (3)
                           +- Filter (2)
                              +- Scan parquet  (1)


(1) Scan parquet 
Output [3]: [pm25#42446, pickup_hour#42449, pickup_date#42448]
Batched: true
Location: PreparedDeltaFileIndex [file:/Users/juozas/Documents/Projects/kth/id2221_labs/id2221-labs/data/lake/gold/integrated_taxi_trips]
PartitionFilters: [isnotnull(pickup_date#4


[VERIFIED] Optimized query produces identical results to baseline.

--- Performance Summary ---
Result Rows:       33
Baseline Time:     0.83s
Optimized Time:    0.20s
Speedup:           75.6%
Storage Overhead:  3.3 KB (1 files)


EVALUATING: Query 4: Zone Weather Demand Variation

--- Baseline Physical Plan (On-Demand) ---
== Physical Plan ==
AdaptiveSparkPlan (20)
+- Sort (19)
   +- Exchange (18)
      +- Project (17)
         +- Filter (16)
            +- HashAggregate (15)
               +- HashAggregate (14)
                  +- HashAggregate (13)
                     +- HashAggregate (12)
                        +- HashAggregate (11)
                           +- HashAggregate (10)
                              +- HashAggregate (9)
                                 +- HashAggregate (8)
                                    +- Project (7)
                                       +- Window (6)
                                          +- Sort (5)
                                       


[VERIFIED] Optimized query produces identical results to baseline.

--- Performance Summary ---
Result Rows:       53
Baseline Time:     3.40s
Optimized Time:    0.22s
Speedup:           93.4%
Storage Overhead:  5.4 KB (1 files)


EVALUATING: Query 5: Peak Travel Hours by Day of Week

--- Baseline Physical Plan (AQE Disabled) ---
== Physical Plan ==
* Project (21)
+- * Sort (20)
   +- Exchange (19)
      +- * Project (18)
         +- * Filter (17)
            +- Window (16)
               +- WindowGroupLimit (15)
                  +- * Sort (14)
                     +- Exchange (13)
                        +- WindowGroupLimit (12)
                           +- * Sort (11)
                              +- * HashAggregate (10)
                                 +- Exchange (9)
                                    +- * HashAggregate (8)
                                       +- * HashAggregate (7)
                                          +- Exchange (6)
                                    

...


--- Optimized Physical Plan (AQE Enabled) ---
== Physical Plan ==
AdaptiveSparkPlan (13)
+- Project (12)
   +- Window (11)
      +- Sort (10)
         +- Exchange (9)
            +- HashAggregate (8)
               +- Exchange (7)
                  +- HashAggregate (6)
                     +- HashAggregate (5)
                        +- Exchange (4)
                           +- HashAggregate (3)
                              +- Project (2)
                                 +- Scan parquet  (1)


(1) Scan parquet 
Output [1]: [pickup_date#42448]
Batched: true
Location: PreparedDeltaFileIndex [file:/Users/juozas/Documents/Projects/kth/id2221_labs/id2221-labs/data/lake/gold/integrated_taxi_trips]
PartitionFilters: [isnotnull(pickup_date#42448)]
ReadSchema: struct<>

(2) Project
Output [2]: [pickup_date#42448, trunc(pickup_date#42448, MM) AS _groupingexpression#49931]
Input [1]: [pickup_date#42448]

(3) HashAggregate
Input [2]: [pickup_date#42448, _groupingexpression#49931]
Keys [2]: [_gr

26/09/26 14:44:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/26 14:44:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/26 14:44:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/26 14:44:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/26 14:44:18 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/26 14:44:18 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/26 1


[VERIFIED] Optimized query produces identical results to baseline.

--- Performance Summary ---
Result Rows:       4
Baseline Time:     0.39s
Optimized Time:    0.22s
Speedup:           41.8%
Storage Overhead:  N/A (AQE Engine Optimization on Integrated Table)



26/09/26 14:44:18 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/26 14:44:18 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/26 14:44:18 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/26 14:44:18 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/26 14:44:18 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/26 14:44:18 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


{'query': 'Query 6: Monthly Trends in Taxi Demand',
 'rows': 4,
 'baseline_s': 0.38513708114624023,
 'optimized_s': 0.22426509857177734,
 'speedup_pct': 41.770058103903594,
 'storage': 'N/A (AQE Engine Optimization on Integrated Table)'}

## 7. Incremental data + schema evolution re-ingest

Generate update files (`data_generator.ipynb`), then re-run bronze → silver → gold.
Extra columns (`humidity`, `aqi`) are allowed by schema validation and flow through silver/gold when present — no YAML rewrite required.

In [8]:
from pyspark.sql import SparkSession
from src.generators.incremental import run_all_generators
from src.spark import create_spark

if True:
    # Recover if an earlier generator call stopped the session
    if SparkSession.getActiveSession() is None:
        print("Spark session was stopped; recreating...")
        spark = create_spark("run-pipeline")

    print("=== Generate incremental updates ===")
    run_all_generators(spark=spark)

    print("\n=== Re-ingest with updates (schema evolution) ===")
    ingest_all(spark, ["taxi_trips", "weather", "air_quality"])
    promote_all(spark, ["taxi_trips", "weather", "air_quality"])

    weather_s = read_delta(spark, SILVER / "weather")
    aq_s = read_delta(spark, SILVER / "air_quality")
    print("silver weather columns:", weather_s.columns)
    print("silver air_quality columns:", aq_s.columns)
    print("humidity present:", "humidity" in weather_s.columns)
    print("aqi present:", "aqi" in aq_s.columns)

    print("\n=== Re-integrate + refresh products ===")
    integrate(spark)
    integrated = read_delta(spark, GOLD / "integrated_taxi_trips")
    print("gold integrated columns:", integrated.columns)
    build_products(spark, force=True)
else:
    print("Skipped incremental demo (RUN_INCREMENTAL=False)")


=== Generate incremental updates ===
Generating taxi trips incremental update...


=== TAXI TRIPS GENERATION REPORT ===
Source directory:       /Users/juozas/Documents/Projects/kth/id2221_labs/id2221-labs/data/raw/taxi_trips/yellow
Source records:         9,554,759
Output update file:     /Users/juozas/Documents/Projects/kth/id2221_labs/id2221-labs/data/raw/taxi_trips/yellow_tripdata_update.parquet
New records written:    669,532
Duplicates injected:    143,322
Original latest pickup: 2024-04-01 00:34:55
New trip period:        2024-04-01 03:34:55 to 2024-07-01 03:33:51
=== WEATHER GENERATION REPORT ===
Source file:            /Users/juozas/Documents/Projects/kth/id2221_labs/id2221-labs/data/raw/weather/weather.csv
Source records:         8,784
Output update file:     /Users/juozas/Documents/Projects/kth/id2221_labs/id2221-labs/data/raw/weather/weather_update.csv
New records written:    88
Time period covered:    2025-01-01 01:00:00 to 2025-01-04 16:00:00
Generating air quality incremental update...
=== AIR QUALITY GENERATION REPORT ===
Source file:            /Users

[bronze/taxi_trips] written (10,367,632 rows)
  + merging update file weather_update.csv
[weather] schema ok=True


[bronze/weather] written (8,872 rows)


  + merging update file hourly_88101_update.csv


...

write by_borough: 7.5s
gold integrated columns: ['taxi_type', 'vendor_id', 'pickup_datetime', 'dropoff_datetime', 'passenger_count', 'trip_distance', 'pickup_location_id', 'pickup_zone', 'pickup_borough', 'dropoff_location_id', 'dropoff_zone', 'dropoff_borough', 'fare_amount', 'tip_amount', 'tolls_amount', 'total_amount', 'temperature_c', 'wind_speed_ms', 'humidity', 'pm25', 'pm25_unit', 'aqi', 'pickup_date', 'pickup_hour']
--- Pipeline Execution Plan (Schema Version: 1.1) ---
[REFRESHING] Product: daily_borough_mobility...


Product daily_borough_mobility refreshed successfully.
[REFRESHING] Product: taxi_zone_monthly_demand...


Product taxi_zone_monthly_demand refreshed successfully.
[REFRESHING] Product: weather_impact_summary...
Product weather_impact_summary refreshed successfully.
[REFRESHING] Product: air_quality_demand_summary...


Product air_quality_demand_summary refreshed successfully.
[REFRESHING] Product: zone_weather_sensitivity...


Product zone_weather_sensitivity refreshed successfully.


## 8. Monitoring report

Ops metrics appended by every bronze/silver/gold step into `data/lake/ops/pipeline_runs`.

Section 9 aggregates those runs **by dataset** (overall across layers + per-layer columns).


In [9]:
import sys
import importlib
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pyspark.sql import SparkSession
from src.spark import create_spark, ensure_runtime
from src.monitoring.recorder import PIPELINE_RUNS
import src.monitoring.queries as monitoring_queries

importlib.reload(monitoring_queries)
from src.monitoring.queries import (
    print_monitoring_report,
    load_pipeline_runs,
    datasets_failing_validation_most,
    datasets_longest_processing,
    rejects_per_execution,
    processing_time_over_executions,
)

ensure_runtime()

if SparkSession.getActiveSession() is None:
    print("Spark session was stopped; recreating for monitoring...")
    spark = create_spark("run-pipeline-monitoring")

print("ops table:", PIPELINE_RUNS)
print_monitoring_report(spark)


ops table: /Users/juozas/Documents/Projects/kth/id2221_labs/id2221-labs/data/lake/ops/pipeline_runs

=== Execution time (each pipeline step) ===
layer    dataset                        time_sec  status
------------------------------------------------------------
bronze   air_quality                       36.63  success
bronze   taxi_trips                         9.68  success
bronze   taxi_zones                         0.55  success
bronze   weather                           10.20  success
silver   air_quality                       14.86  success
silver   taxi_trips                        32.89  success
silver   taxi_zones                         1.74  success
silver   weather                           12.12  success
gold     integrated_taxi_trips             27.80  success
gold     daily_borough_mobility             4.17  success
gold     taxi_zone_monthly_demand           2.60  success
gold     weather_impact_summary             1.83  success
gold     air_quality_demand_summary      

## 9. Monitoring Spark SQL questions

Answered from `pipeline_runs` (same helpers as `src.monitoring.queries`).

Each answer is **one row per source dataset family** (`taxi_trips`, `weather`,
`air_quality`, `taxi_zones`, …). Gold products like `integrated_taxi_trips` and
`weather_impact_summary` are rolled into their source family, with overall
aggregates plus `bronze_*` / `silver_*` / `gold_*` columns.


### Which dataset fails validation most frequently? (overall + per layer)


In [10]:
import sys
import importlib
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pyspark.sql import SparkSession
from src.spark import create_spark, ensure_runtime
import src.monitoring.queries as monitoring_queries

importlib.reload(monitoring_queries)
from src.monitoring.queries import (
    load_pipeline_runs,
    datasets_failing_validation_most,
    datasets_longest_processing,
    rejects_per_execution,
    processing_time_over_executions,
)

ensure_runtime()
if SparkSession.getActiveSession() is None:
    spark = create_spark("run-pipeline-monitoring")

load_pipeline_runs(spark)
datasets_failing_validation_most(spark).show(50, truncate=False)


+-----------+-----------+-------------------------+----------------------+------------------+--------------------------+-----------------------+------------------+--------------------------+-----------------------+----------------+------------------------+---------------------+
|dataset    |failed_runs|total_validation_failures|total_rejected_records|bronze_failed_runs|bronze_validation_failures|bronze_rejected_records|silver_failed_runs|silver_validation_failures|silver_rejected_records|gold_failed_runs|gold_validation_failures|gold_rejected_records|
+-----------+-----------+-------------------------+----------------------+------------------+--------------------------+-----------------------+------------------+--------------------------+-----------------------+----------------+------------------------+---------------------+
|taxi_trips |2          |667032                   |667032                |0                 |0                         |0                      |2                 |

### Which dataset requires the longest processing time? (overall + per layer)


In [11]:
datasets_longest_processing(spark).show(50, truncate=False)


+-----------+----------------------+----------------------+------------------------+---------------------+----+--------------+--------------+-----------+--------------+--------------+-----------+------------+------------+---------+
|dataset    |avg_execution_time_sec|max_execution_time_sec|total_execution_time_sec|avg_processed_records|runs|bronze_avg_sec|bronze_max_sec|bronze_runs|silver_avg_sec|silver_max_sec|silver_runs|gold_avg_sec|gold_max_sec|gold_runs|
+-----------+----------------------+----------------------+------------------------+---------------------+----+--------------+--------------+-----------+--------------+--------------+-----------+------------+------------+---------+
|taxi_trips |14.06                 |32.89                 |168.74                  |4925346.0            |12  |12.23         |14.77         |2          |30.57         |32.89         |2          |10.39       |27.8        |8        |
|air_quality|14.8                  |36.63                 |88.79        

### How many records were rejected (by dataset, overall + per layer)?


In [12]:
rejects_per_execution(spark).show(100, truncate=False)


+-----------+----+-----------------------+----------------------+----------------------+-------------------------+----------------+---------------+---------------+----------------+---------------+---------------+--------------+-------------+-------------+
|dataset    |runs|total_processed_records|total_inserted_records|total_rejected_records|total_validation_failures|bronze_processed|bronze_inserted|bronze_rejected|silver_processed|silver_inserted|silver_rejected|gold_processed|gold_inserted|gold_rejected|
+-----------+----+-----------------------+----------------------+----------------------+-------------------------+----------------+---------------+---------------+----------------+---------------+---------------+--------------+-------------+-------------+
|taxi_trips |12  |59104148               |58437116              |667032                |667032                   |19922410        |19922410       |0              |19922410        |19255378       |667032         |19259328      |19259

### How has processing time changed (by dataset, overall + per layer)?


In [13]:
processing_time_over_executions(spark).show(100, truncate=False)


+-----------+----+----------------------+----------------------+----------------------+------------------------+------------------------+-----------------------+--------------+--------------+------------+--------------------------+--------------------------+
|dataset    |runs|avg_execution_time_sec|min_execution_time_sec|max_execution_time_sec|total_execution_time_sec|first_execution_time_sec|last_execution_time_sec|bronze_avg_sec|silver_avg_sec|gold_avg_sec|first_started_at          |last_started_at           |
+-----------+----+----------------------+----------------------+----------------------+------------------------+------------------------+-----------------------+--------------+--------------+------------+--------------------------+--------------------------+
|taxi_trips |12  |14.06                 |2.09                  |32.89                 |168.74                  |9.68                    |7.42                   |12.23         |30.57         |10.39       |2026-09-26 10:41:10

## 10. Production readiness metrics

Instrumentation from `src.benchmark.production_readiness`:

| Metric | What is measured |
|---|---|
| Incremental update time | generate → bronze → silver → gold integrate |
| Analytical refresh time | `build_products(force=True)` |
| Storage overhead | lake (bronze/silver/gold/ops) vs raw; products, rejects, dual layout |
| Validation overhead | wall time of schema + row DQ checks (no write) |
| Monitoring overhead | `pipeline_runs` Delta append cost |

Set `RUN_PROD_INCREMENTAL=False` to skip re-running the incremental pipeline and only
measure validation / monitoring / storage / analytical refresh.


In [14]:
import importlib
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pyspark.sql import SparkSession
from src.spark import create_spark, ensure_runtime
import src.benchmark.production_readiness as prod_ready

importlib.reload(prod_ready)
from src.benchmark.production_readiness import evaluate_production_readiness

ensure_runtime()
if SparkSession.getActiveSession() is None:
    print("Spark session was stopped; recreating for production readiness...")
    spark = create_spark("run-pipeline-prod-ready")

# If section 7 already ran generators + re-ingest, set False to avoid a full redo.
RUN_PROD_INCREMENTAL = True
# Section 7 usually already wrote update files — skip regenerate by default.
RUN_PROD_GENERATE = False

prod_report = evaluate_production_readiness(
    spark,
    run_incremental=RUN_PROD_INCREMENTAL,
    generate=RUN_PROD_GENERATE,
    run_analytical_refresh=True,
    measure_validation=True,
    measure_monitoring=True,
    monitoring_samples=3,
)

prod_report.to_dict()



=== Measuring incremental update time ===
  + merging update file yellow_tripdata_update.parquet
[taxi_trips/yellow] schema ok=True


[bronze/taxi_trips] written (10,367,632 rows)
  + merging update file weather_update.csv
[weather] schema ok=True


[bronze/weather] written (8,872 rows)


  + merging update file hourly_88101_update.csv


[air_quality] schema ok=True


...

Product air_quality_demand_summary refreshed successfully.
[REFRESHING] Product: zone_weather_sensitivity...


Product zone_weather_sensitivity refreshed successfully.
Analytical refresh: 17.93s

=== Measuring validation overhead ===
  + merging update file yellow_tripdata_update.parquet
[taxi_trips] schema ok=True


[validation overhead] taxi_trips: 6.52s (good=9,837,995 rejects=529,637)
  + merging update file weather_update.csv
[weather] schema ok=True
[validation overhead] weather: 0.67s (good=8,872 rejects=0)


  + merging update file hourly_88101_update.csv
[air_quality] schema ok=True
[validation overhead] air_quality: 1.37s (good=112,838 rejects=4,600)

=== Measuring monitoring overhead ===
[monitoring overhead] 3 appends → 1.798s total, 0.599s/append, ~7.19s for 12 pipeline steps
  ops table: /Users/juozas/Documents/Projects/kth/id2221_labs/id2221-labs/data/lake/ops/pipeline_runs

PRODUCTION READINESS METRICS

1) Incremental update time
   total: 110.70s
   - bronze_ingest          39.10s
   - silver_promote         48.35s
   - gold_integrate         23.24s

2) Analytical refresh time
   gold data products (force refresh): 17.93s

3) Storage overhead (updated platform)
   raw landing zone:        2430.64 MB
   bronze:                   680.11 MB
   silver:                   763.15 MB
   gold (all):              1431.12 MB
     data products:            2.03 MB
     dual borough layout:    709.81 MB
   rejects:                   31.50 MB
   ops (monitoring):           0.33 MB
   lake total

{'incremental_update_sec': 110.696,
 'incremental_phases': [{'name': 'bronze_ingest', 'seconds': 39.097},
  {'name': 'silver_promote', 'seconds': 48.355},
  {'name': 'gold_integrate', 'seconds': 23.244}],
 'analytical_refresh_sec': 17.93,
 'storage_before_mb': {'raw_mb': 2430.64,
  'bronze_mb': 438.64,
  'silver_mb': 502.89,
  'gold_mb': 943.42,
  'ops_mb': 0.19,
  'products_mb': 1.24,
  'rejects_mb': 17.51,
  'dual_layout_mb': 468.51,
  'lake_mb': 1885.14,
  'platform_overhead_mb': 1885.14,
  'file_count': 7689.0},
 'storage_after_mb': {'raw_mb': 2430.64,
  'bronze_mb': 680.11,
  'silver_mb': 763.15,
  'gold_mb': 1431.12,
  'ops_mb': 0.33,
  'products_mb': 2.03,
  'rejects_mb': 31.5,
  'dual_layout_mb': 709.81,
  'lake_mb': 2874.71,
  'platform_overhead_mb': 2874.71,
  'file_count': 12072.0},
 'storage_growth_mb': 989.57,
 'validation_overhead_sec': 8.554,
 'validation_by_dataset_sec': {'taxi_trips': 6.523,
  'weather': 0.665,
  'air_quality': 1.366},
 'monitoring_overhead_sec': 1.798

### Results summary (structured)

Headline numbers for the five production-readiness checks:


In [15]:
from IPython.display import Markdown, display

d = prod_report.to_dict()
after = d["storage_after_mb"] or {}
raw_mb = after.get("raw_mb") or 0
lake_mb = after.get("lake_mb") or 0
ratio = (lake_mb / raw_mb) if raw_mb else float("nan")

rows = [
    ("Incremental update time", f"{d['incremental_update_sec']:.2f}s"),
    ("Analytical refresh time", f"{d['analytical_refresh_sec']:.2f}s"),
    (
        "Storage overhead (lake)",
        f"{lake_mb:.2f} MB lake vs {raw_mb:.2f} MB raw ({ratio:.2f}x); "
        f"growth={d['storage_growth_mb']:.2f} MB",
    ),
    ("Validation overhead", f"{d['validation_overhead_sec']:.2f}s"),
    (
        "Monitoring overhead",
        f"{d['monitoring_per_append_sec']:.3f}s/append, "
        f"~{d['monitoring_estimated_pipeline_sec']:.2f}s / full pipeline",
    ),
]

md = "| Metric | Result |\n|---|---|\n"
md += "\n".join(f"| {k} | {v} |" for k, v in rows)
display(Markdown(md))

print("\nPhase breakdown (incremental):")
for phase in d["incremental_phases"]:
    print(f"  {phase['name']:<22} {phase['seconds']:.2f}s")

print("\nValidation by dataset:")
for name, sec in d["validation_by_dataset_sec"].items():
    print(f"  {name:<22} {sec:.2f}s")

print("\nStorage breakdown (MB):")
for key in (
    "raw_mb",
    "bronze_mb",
    "silver_mb",
    "gold_mb",
    "products_mb",
    "dual_layout_mb",
    "rejects_mb",
    "ops_mb",
    "lake_mb",
):
    print(f"  {key:<18} {after.get(key, 0):>10.2f}")

spark.stop()
print("\nFull pipeline demo finished.")


| Metric | Result |
|---|---|
| Incremental update time | 110.70s |
| Analytical refresh time | 17.93s |
| Storage overhead (lake) | 2874.71 MB lake vs 2430.64 MB raw (1.18x); growth=989.57 MB |
| Validation overhead | 8.55s |
| Monitoring overhead | 0.599s/append, ~7.19s / full pipeline |


Phase breakdown (incremental):
  bronze_ingest          39.10s
  silver_promote         48.35s
  gold_integrate         23.24s

Validation by dataset:
  taxi_trips             6.52s
  weather                0.67s
  air_quality            1.37s

Storage breakdown (MB):
  raw_mb                2430.64
  bronze_mb              680.11
  silver_mb              763.15
  gold_mb               1431.12
  products_mb              2.03
  dual_layout_mb         709.81
  rejects_mb              31.50
  ops_mb                   0.33
  lake_mb               2874.71

Full pipeline demo finished.
